# Tính tay và giải thích tham số

Ví dụ tự đặt giúp hiểu attention, sigmoid, BCE và gradient. Số minh họa không phải kết quả GoEmotions. Notebook không lập thêm thí nghiệm LR/epoch và không huấn luyện.

In [1]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from examples.pretrained_math import calculate
calculate()

{'scope': 'two-label toy, not benchmark or pretrained inference',
 'attention_scores': [0.7071067811865475, 0.0],
 'attention_weights': [0.6697615493266569, 0.3302384506733431],
 'attention_output': [0.6697615493266569, 0.3302384506733431],
 'logits': [0.9, -0.4],
 'probabilities': [0.7109495026250039, 0.401312339887548],
 'mean_bce': 0.4270845635660202,
 'gradient_logits': [-0.14452524868749805, 0.200656169943774],
 'gradient_weight_gratitude': [-0.08671514921249883, 0.028905049737499612],
 'sgd_weight_new': [1.2000017343029843, -0.40000057810099476],
 'sgd_bias_new': 0.10000289050497375,
 'finite_difference_check_passed': True,
 'micro_batches_per_epoch': 2714}

## Cấu hình thực tế của ba kiến trúc
Learning rate và epoch là cấu hình đã chạy, chưa có ablation chứng minh tối ưu. Seed điều khiển khởi tạo head, dropout và shuffle; số 42, 123 hay 2026 không có ưu thế riêng.

In [2]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src').is_dir())
EXPERIMENT = ROOT/'experiments/best_epochs_20261103'
MAIN = EXPERIMENT/'main_results'
lock = json.loads((MAIN/'LOCK_BEFORE_TEST.json').read_text(encoding='utf-8'))
configurations = []
for run in lock['runs']:
    folder = MAIN/'training_receipts'/run['architecture']/f"seed_{run['seed']}"
    metadata = json.loads((folder/'run_metadata.json').read_text(encoding='utf-8'))
    config = metadata['config']
    configurations.append(dict(architecture=run['architecture'], seed=run['seed'],
        selected_epoch=run['selected_epoch'], learning_rate=config['learning_rate'],
        epochs=config['epochs'], batch_size=config['batch_size'], max_length=config['max_length']))
assert len(configurations) == 9
display(pd.DataFrame(configurations))
print('Cấu hình thực của chín lượt hiện tại; không tải weights và không huấn luyện lại.')


,architecture,seed,selected_epoch,learning_rate,epochs,batch_size,max_length
0,bert,42,7,0.00002,8,16,128
1,bert,123,6,0.00002,8,16,128
2,bert,2026,7,0.00002,8,16,128
3,roberta,42,7,0.00002,8,16,128
4,roberta,123,8,0.00002,8,16,128
5,roberta,2026,5,0.00002,8,16,128
6,distilbert,42,7,0.00002,8,16,128
7,distilbert,123,8,0.00002,8,16,128
8,distilbert,2026,8,0.00002,8,16,128


Cấu hình thực của chín lượt hiện tại; không tải weights và không huấn luyện lại.


## Cách đọc chẩn đoán
Loss dao động giữa mini-batch chưa chứng minh mô hình thất bại. Cần đọc xu hướng train/validation, Macro-F1 và luật ngưỡng. Gap nhỏ có thể do cả train lẫn validation đều thấp; epoch lớn có thể làm quá khớp, LR quá nhỏ có thể học chưa đủ. Các nhận xét phải gắn với số đo, không kết luận nhân quả từ một checkpoint.

In [3]:
audit = pd.read_csv(EXPERIMENT/'overfit_audit/overfit_comparison.csv')
columns = ['architecture','seed','selected_epoch','old_train_bce','old_val_bce',
           'selected_train_bce','selected_val_bce','old_gap_macro_f1','selected_gap_macro_f1']
display(audit[columns].round(6))
print('Đo train/validation ở eval mode, AMP fp16; đọc F1 cùng BCE để nhận xét overfit.')


,architecture,seed,selected_epoch,old_train_bce,old_val_bce,selected_train_bce,selected_val_bce,old_gap_macro_f1,selected_gap_macro_f1
0,bert,42,7,0.035716,0.094315,0.025870,0.106145,0.264199,0.333544
1,bert,123,6,0.038038,0.091837,0.030087,0.101499,0.250426,0.304761
2,bert,2026,7,0.036805,0.093245,0.024566,0.105355,0.249484,0.336083
3,roberta,42,7,0.070111,0.083892,0.038351,0.094248,0.059863,0.247892
4,roberta,123,8,0.071099,0.083959,0.036485,0.097775,0.063906,0.263011
5,roberta,2026,5,0.069696,0.083688,0.048991,0.088151,0.067250,0.190798
6,distilbert,42,7,0.068955,0.084445,0.029648,0.105085,0.069215,0.298411
7,distilbert,123,8,0.069196,0.084083,0.028149,0.106701,0.065563,0.306095
8,distilbert,2026,8,0.068971,0.084298,0.027956,0.105625,0.069229,0.305243


Đo train/validation ở eval mode, AMP fp16; đọc F1 cùng BCE để nhận xét overfit.
